# Module 3 — AskUserQuestion (clarification + session resume)

A good analyst doesn't guess at an ambiguous request — they ask. *"Show me the top students"* —
top by **GPA**? by **credits**? for **which semester**? In this module the agent asks a
**clarification question** before writing SQL, you answer, and it continues — across multiple
rounds if needed.

The twist: AgentCore is **serverless**, so the deployed agent can't pause and wait for terminal
input. Instead it emits its questions as a structured JSON block and returns; a small driver
collects your answers and **re-invokes with session resume** so the agent picks up where it left off.


## It's the same agent — one flag flips it on

No new agent file. `build_agent_options()` (the single source of truth from Module 1) takes an
`enable_clarification` flag:

```python
build_agent_options(request_id=rid, enable_clarification=True, can_use_tool=...)
```

That adds the **AskUserQuestion** tool and a short clarification instruction to the system prompt.
The deploy entrypoint (`agent_agentcore.py`) sets it, plus:
- a `can_use_tool` callback that **allows** AskUserQuestion (it doesn't try to answer it — serverless),
- detects the AskUserQuestion call in the stream → emits a `clarification_needed` JSON block (with the
  SDK **session id**) → returns,
- accepts `claude_agent_sdk_session_id` in the payload to **resume** a prior session.

A drift-guard test confirms this module's `agent.py` is still byte-identical to Module 1's — the
clarification behavior is an *override*, not a fork.

## Setup

### Python environment

Please make you had followeed the setup instruction in  **Module 0's Setup section**  (`uv sync` from the project root + select the `.venv` kernel) to install the depandency. In this notebook, you can just select .venv kernel

![](images/select-venv.png)

![](images/venv-selected.png)

### Install Node.js + AgentCore CLI

Run the cell below to install the AgentCore CLI and CDK dependencies (required for deployment).

In [ ]:
!bash setup.sh

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()
import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")
with open("agentcore/aws-targets.json", "w") as f:
    json.dump([{"name": "default", "account": acct, "region": region}], f, indent=2)
print("target:", acct, region)

## Deploy

Same lifecycle as Module 2 — the runtime is configured with `enableOtel` and the Athena/Glue/S3
permissions, and it points at this module's `analytics_agent/` bundle (the clarification-enabled
entrypoint).

In [ ]:
!agentcore deploy -y

## Try an ambiguous question directly

Invoke once with a deliberately ambiguous prompt. Instead of an answer, you'll get a
`clarification_needed` JSON block — the agent is asking *you* to pick. Note the
`claude_agent_sdk_session_id` in the output; that's what lets the next call resume.

In [ ]:
!agentcore invoke '{"prompt": "Show me the top students."}' \
    --runtime analytics --session-id agentic-analytics-m3-demo-session-0001

## Let the driver close the loop (multi-round)

`scripts/invoke_agentcore.py` automates the round-trip: it invokes, detects the
`clarification_needed` block, prompts you for answers in the terminal, then **re-invokes with the
session id** to resume — repeating until the agent gives a final answer.

Because it reads your answers interactively, run it in a **terminal**, from the repo root `cd` into
this module's folder `advanced/agentic-analytics/module-3-follow-up` first (not a notebook cell):

```bash
cd advanced/agentic-analytics/module-3-follow-up
uv run python scripts/invoke_agentcore.py "Show me the top students"
```

You'll see something like:
```
The agent needs clarification:
  Rank by: Which metric should rank "top" students?
    1. GPA — academic standing
    2. Credits earned — progress toward graduation
Your choice (number, comma-separated, or free text): 1
▶ invoking (resume)…
The top 10 students by GPA are…
✅ Done — the agent answered.
```

in the terminal, once running this script, the agent will you clarification question and you can type in your answer and it may look below (screenshots)


![](images/follow-up-question-1.png)

![](images/follow-up-question-2.png)

## See the trace + clean up

The clarification rounds are traced in CloudWatch just like Module 2 (observability is on).

In [ ]:
!agentcore traces list --runtime analytics --since 1h

## Recap

- **Clarification is one flag** on the shared `build_agent_options()` — `enable_clarification=True`
  adds AskUserQuestion. The agent's `agent.py` never forked (drift-guard proves it).
- Serverless multi-round works by **emitting questions as JSON + resuming the SDK session**, driven
  by `scripts/invoke_agentcore.py`.
- Everything else — Athena tool, skills, deploy, observability — is inherited from Modules 1–2.

That completes the Agentic Analytics track: **set up the data (M0) → build the agent (M1) → deploy
& observe it (M2) → make it ask good follow-up questions (M3).**